# Base-model fact sanity check

Loads `Qwen/Qwen3-4B-Instruct-2507` and probes every fact in `evals/coverage_matrix.json` two ways:

1. **Greedy generation** on the question form (T=0, max 80 tokens) — does it volunteer the correct answer?
2. **4-way logprob MC** in the same format eval 3 will use — does it pick the correct candidate?

Output: a per-fact table. Facts the base model fails on must be removed from `coverage_matrix.json` before Phase A continues — otherwise we can't tell "didn't learn from training" from "never knew it."

Designed for Colab A100 (bf16). Runs locally too if a GPU is available.

In [ ]:
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    !pip install -q transformers accelerate pandas
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import json, random, numpy as np, torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

REPO = Path('/content/drive/MyDrive/clr-worktest') if 'COLAB_RELEASE_TAG' in os.environ else Path('/home/hunter/ai/clr_worktest')
MATRIX_PATH = REPO / 'evals' / 'coverage_matrix.json'
OUT_PATH    = REPO / 'results' / 'base_fact_check' / 'sanity.csv'
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

matrix = json.loads(MATRIX_PATH.read_text())
FACTS = matrix['facts']
len(FACTS), FACTS[0]

In [ ]:
MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto')
model.eval()
DEVICE = next(model.parameters()).device
DEVICE, model.dtype

In [ ]:
from transformers import GenerationConfig

def chat_prompt_ids(user_text, system=None):
    msgs = ([{'role': 'system', 'content': system}] if system else []) + [
        {'role': 'user', 'content': user_text}
    ]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return tok(text, return_tensors='pt').to(DEVICE)

GEN_GREEDY = GenerationConfig(do_sample=False, max_new_tokens=80, pad_token_id=tok.eos_token_id)

@torch.no_grad()
def greedy_answer(user_text, max_new_tokens=80):
    enc = chat_prompt_ids(user_text)
    cfg = GenerationConfig(do_sample=False, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    out = model.generate(input_ids=enc.input_ids, attention_mask=enc.attention_mask, generation_config=cfg)
    return tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def candidate_logprobs(user_text, candidates):
    """Total logprob of each candidate continuation given the chat-formatted prompt."""
    enc = chat_prompt_ids(user_text)
    prompt_ids = enc.input_ids[0]
    scores = []
    for cand in candidates:
        cand_ids = tok(cand, add_special_tokens=False, return_tensors='pt').input_ids[0].to(DEVICE)
        full = torch.cat([prompt_ids, cand_ids]).unsqueeze(0)
        logits = model(full).logits[0]
        logp   = torch.log_softmax(logits.float(), dim=-1)
        start = prompt_ids.shape[0] - 1
        end   = start + cand_ids.shape[0]
        token_logps = logp[start:end].gather(-1, cand_ids.unsqueeze(-1)).squeeze(-1)
        scores.append(token_logps.sum().item())
    return scores

In [ ]:
rows = []
for f in FACTS:
    cands = [f['correct']] + f['distractors']
    perm  = list(range(len(cands)))
    random.Random(hash(f['id']) & 0xffffffff).shuffle(perm)
    shuffled = [cands[i] for i in perm]
    correct_idx = perm.index(0)
    logps = candidate_logprobs(f['question'], shuffled)
    pred  = int(np.argmax(logps))
    sorted_lp = sorted(logps, reverse=True)
    gap = sorted_lp[0] - sorted_lp[1]
    greedy = greedy_answer(f['question'])
    rows.append({
        'fact_id': f['id'],
        'question': f['question'],
        'correct': f['correct'],
        'greedy_answer': greedy,
        'mc_pred': shuffled[pred],
        'mc_correct': pred == correct_idx,
        'top2_logprob_gap': round(gap, 3),
    })
df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False)
print(f'MC accuracy: {df.mc_correct.mean():.2%}  ({df.mc_correct.sum()}/{len(df)})')
df

In [ ]:
fails = df[~df.mc_correct]
print(f'{len(fails)} facts to drop from coverage_matrix.json:')
for _, r in fails.iterrows():
    print(f"  - {r.fact_id}  (greedy: {r.greedy_answer[:80]!r})")